In [1]:
import os

# Fixes for loading datasets on windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["DATASETS_VERBOSITY"] = "error"
os.environ["WANDB_DISABLED"] = "true" # Keeps logs clean

import evaluate
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

print("Libraries loaded safely. GPU Available:", torch.cuda.is_available())

Libraries loaded safely. GPU Available: True


In [2]:
print("Bypassing Hugging Face Hub and downloading raw JSONL files from GitHub...")

# URL extracted from the author's deprecated script
base_url = "https://raw.githubusercontent.com/fhamborg/NewsMTSC/6b838e00f54423c253806327a0ae24dbffa24c9e/NewsSentiment/experiments/default/datasets/newsmtsc-rw-hf/"

data_files = {
    "train": base_url + "train.jsonl",
    "validation": base_url + "dev.jsonl",
    "test": base_url + "test.jsonl"
}

# Use the native, secure JSON loader
dataset = load_dataset("json", data_files=data_files)

print("\n--- Dataset Splits ---")
print(dataset)

print("\n--- Sample Record From Training Set ---")
print(dataset['train'][0])

Bypassing Hugging Face Hub and downloading raw JSONL files from GitHub...

--- Dataset Splits ---
DatasetDict({
    train: Dataset({
        features: ['mention', 'polarity', 'from', 'to', 'sentence', 'id'],
        num_rows: 8739
    })
    validation: Dataset({
        features: ['mention', 'polarity', 'from', 'to', 'sentence', 'id'],
        num_rows: 343
    })
    test: Dataset({
        features: ['mention', 'polarity', 'from', 'to', 'sentence', 'id'],
        num_rows: 803
    })
})

--- Sample Record From Training Set ---
{'mention': 'Winner', 'polarity': 0, 'from': 0, 'to': 6, 'sentence': 'Winner wrote that she had a 30-minute private meeting with the Republican lawmaker’s state policy director.', 'id': 'allsides_1000_401_25_Reality Leigh Winner_0_6'}


In [3]:
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_absa_function(examples):
    tokenized_inputs = tokenizer(
        examples["sentence"], 
        examples["mention"], 
        truncation=True, 
        max_length=128
    )
    # Shift labels: -1, 0, 1 -> 0, 1, 2
    tokenized_inputs["labels"] = [label + 1 for label in examples["polarity"]]
    return tokenized_inputs

print("Tokenizing entire dataset...")
tokenized_datasets = dataset.map(tokenize_absa_function, batched=True)
print("Tokenization Complete!")

Tokenizing entire dataset...
Tokenization Complete!


In [4]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    return f1_metric.compute(predictions=predictions, references=labels, average="macro")

print("Model and metrics engine successfully initialized.")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | Details
----------------------------+------------+--------
lm_head.layer_norm.bias     | UNEXPECTED |        
lm_head.bias                | UNEXPECTED |        
roberta.pooler.dense.bias   | UNEXPECTED |        
lm_head.dense.weight        | UNEXPECTED |        
lm_head.layer_norm.weight   | UNEXPECTED |        
lm_head.dense.bias          | UNEXPECTED |        
roberta.pooler.dense.weight | UNEXPECTED |        
classifier.out_proj.bias    | MISSING    |        
classifier.dense.weight     | MISSING    |        
classifier.out_proj.weight  | MISSING    |        
classifier.dense.bias       | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model and metrics engine successfully initialized.


In [5]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="../checkpoints",
    learning_rate=2.6671030576600103e-05,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.036762735036943384,
    warmup_steps=58,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print("Training configuration ready.")

Training configuration ready.


In [6]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.519585,0.458290,0.804368
2,0.401913,0.436795,0.830921
3,0.223939,0.460207,0.867028
4,0.185098,0.635971,0.859967
5,0.126200,0.738715,0.847121


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2735, training_loss=0.3024380207933499, metrics={'train_runtime': 117.0331, 'train_samples_per_second': 373.356, 'train_steps_per_second': 23.369, 'total_flos': 874894466842992.0, 'train_loss': 0.3024380207933499, 'epoch': 5.0})

In [7]:
output_model_path = "../newsmtsc_distilroberta_absa"
trainer.save_model(output_model_path)
tokenizer.save_pretrained(output_model_path)
print(f"Model successfully saved to {output_model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to ../newsmtsc_distilroberta_absa
